# <font color="#418FDE" size="6.5" uppercase>**Distances Kernels**</font>

>Last update: 20260713.
    
By the end of this Lecture, you will be able to:
- Compute pairwise distances and similarities for small datasets. 
- Compare common distance metrics and kernel functions visually and numerically. 
- Use precomputed distance or kernel matrices with compatible estimators. 


## **1. Distance Metrics**

### **1.1. Pairwise Distance Matrix**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_A/image_01_01.jpg?v=1783956264" width="250">



>* Shows distances between every observation pair
>* Reveals symmetry, zeros, and natural groups

>* Choose features and a suitable distance metric
>* Store pairwise comparisons, not original measurements

>* Scale affects distance interpretation
>* Matrices reveal patterns for distance-based methods



In [ ]:
#@title Python Code - Pairwise Distance Matrix

# Pairwise distances compare every observation pair.
# Small matrices reveal similarity patterns quickly.
# Feature scaling affects distance interpretation strongly.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create a tiny apartment dataset.
apartments = pd.DataFrame(
    {
        "size_sqft": [650, 700, 1200, 1250],
        "distance_miles": [1.2, 1.5, 6.0, 6.5],
    },
    index=["A", "B", "C", "D"],
)

# Convert values into a numeric array.
X = apartments.to_numpy(dtype=float)

# Validate the small matrix shape.
assert X.shape == (4, 2), "Expected four rows and two features."

# Compute all row differences using broadcasting.
differences = X[:, None, :] - X[None, :, :]

# Convert differences into Euclidean distances.
distances = np.sqrt(np.sum(differences ** 2, axis=2))

# Store distances with readable labels.
distance_table = pd.DataFrame(
    np.round(distances, 2),
    index=apartments.index,
    columns=apartments.index,
)

# Print compact teaching output.
print("Apartment features:")
print(apartments)
print("\nPairwise Euclidean distance matrix:")
print(distance_table)

# Plot one heatmap for visual comparison.
fig, ax = plt.subplots(figsize=(5, 4))
image = ax.imshow(distances, cmap="Blues")

# Add labels and a colorbar.
ax.set_xticks(range(len(apartments.index)))
ax.set_yticks(range(len(apartments.index)))
ax.set_xticklabels(apartments.index)
ax.set_yticklabels(apartments.index)

# Add a clear plot title.
ax.set_title("Pairwise Distance Matrix")
fig.colorbar(image, ax=ax, label="Euclidean distance")

# Display the finished visualization.
plt.tight_layout()
plt.show()



### **1.2. Euclidean Manhattan**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_A/image_01_02.jpg?v=1783956266" width="250">



>* Euclidean measures straight-line feature separation
>* Manhattan sums feature-by-feature differences

>* Euclidean distance emphasizes large feature gaps
>* Manhattan distance sums feature-by-feature changes

>* Scale features before comparing distances
>* Different metrics shape similarity results



In [ ]:
#@title Python Code - Euclidean Manhattan

# Compare two common distance metrics.
# Use tiny apartment feature data.
# Visualize pairwise distances as heatmaps.

# Import numerical and plotting tools.
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Create three apartment observations.
names = ["A", "B", "C"]
features = np.array([[650, 2, 0.8], [700, 2, 1.1], [900, 3, 2.0]])

# Standardize features for fair distances.
means = features.mean(axis=0)
stds = features.std(axis=0)
scaled = (features - means) / stds

# Validate the tiny matrix shape.
assert scaled.shape == (3, 3)
assert np.all(stds > 0)

# Compute pairwise coordinate differences.
diffs = scaled[:, None, :] - scaled[None, :, :]
euclidean = np.sqrt(np.sum(diffs ** 2, axis=2))
manhattan = np.sum(np.abs(diffs), axis=2)

# Print a compact numerical comparison.
print("Pair A-B Euclidean:", round(euclidean[0, 1], 3))
print("Pair A-B Manhattan:", round(manhattan[0, 1], 3))
print("Pair A-C Euclidean:", round(euclidean[0, 2], 3))
print("Pair A-C Manhattan:", round(manhattan[0, 2], 3))

# Draw both distance matrices together.
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
sns.heatmap(euclidean, annot=True, cmap="Blues", xticklabels=names, yticklabels=names, ax=axes[0])
sns.heatmap(manhattan, annot=True, cmap="Oranges", xticklabels=names, yticklabels=names, ax=axes[1])

# Add clear titles for learners.
axes[0].set_title("Euclidean distance")
axes[1].set_title("Manhattan distance")
plt.tight_layout()
plt.show()



### **1.3. Custom Distance Metrics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_A/image_01_03.jpg?v=1783956269" width="250">



>* Standard distances may miss real similarity
>* Custom metrics encode domain knowledge

>* Apply tailored comparisons to every data pair
>* Capture meaningful similarity beyond raw values

>* Balance feature influence and data scales
>* Inspect distances to validate meaningful similarity



In [ ]:
#@title Python Code - Custom Distance Metrics

# Custom metrics encode domain knowledge.
# Small matrices make distances inspectable.
# This example compares apartment similarity.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create a tiny apartment dataset.
apartments = pd.DataFrame({
    "name": ["A", "B", "C", "D"],
    "rent_dollars": [1200, 1500, 1250, 2100],
    "size_sqft": [650, 700, 900, 1100],
    "transit_miles": [0.2, 1.5, 0.4, 2.0],
    "has_parking": [1, 0, 1, 1],
})

# Validate the small dataset shape.
assert apartments.shape == (4, 5)

# Select numeric comparison columns.
features = apartments[[
    "rent_dollars", "size_sqft", "transit_miles", "has_parking"
]].to_numpy(dtype=float)

# Scale columns into comparable ranges.
mins = features.min(axis=0)
ranges = np.ptp(features, axis=0)
ranges = np.where(ranges == 0, 1, ranges)
scaled = (features - mins) / ranges

# Define a domain-aware custom distance.
def apartment_distance(row_a, row_b):
    rent_gap = abs(row_a[0] - row_b[0])
    size_gap = abs(row_a[1] - row_b[1])
    transit_gap = abs(row_a[2] - row_b[2])
    parking_gap = abs(row_a[3] - row_b[3])

    return 0.45 * rent_gap + 0.20 * size_gap + 0.25 * transit_gap + 0.10 * parking_gap

# Compute every pairwise custom distance.
n_items = scaled.shape[0]
distances = np.zeros((n_items, n_items))
for i in range(n_items):
    for j in range(n_items):
        distances[i, j] = apartment_distance(scaled[i], scaled[j])

# Convert distances into similarities.
similarities = 1 / (1 + distances)
labels = apartments["name"].tolist()

# Print compact numerical results.
print("Custom distance matrix:")
print(pd.DataFrame(distances, labels, labels).round(2))
print("\nClosest pair by custom distance:")
masked = distances + np.eye(n_items) * 999
best_i, best_j = np.unravel_index(masked.argmin(), masked.shape)
print(f"Apartments {labels[best_i]} and {labels[best_j]} are closest.")

# Plot one heatmap for visual comparison.
plt.figure(figsize=(5, 4))
plt.imshow(similarities, cmap="YlGnBu", vmin=0, vmax=1)
plt.xticks(range(n_items), labels)
plt.yticks(range(n_items), labels)
plt.colorbar(label="custom similarity")
plt.title("Custom Apartment Similarity")
plt.tight_layout()
plt.show()



## **2. Similarity Kernels**

### **2.1. Cosine Similarity**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_A/image_02_01.jpg?v=1783956257" width="250">



>* Compares vector direction, not overall size
>* Angle reveals similarity beyond Euclidean distance

>* Higher cosine means more similar patterns
>* Compares proportions, not total activity

>* Cosine matrices reveal direction-based groups.
>* Highlights orientation, but ignores useful magnitude.



In [ ]:
#@title Python Code - Cosine Similarity

# Compare vectors using cosine similarity.
# Magnitude changes should not dominate.
# Heatmaps reveal directional similarity patterns.

import numpy as np
import matplotlib.pyplot as plt

# Create tiny profile vectors.
labels = ["short review", "long review", "sports review", "opposite tone"]
X = np.array([
    [2.0, 3.0, 1.0],
    [20.0, 30.0, 10.0],
    [1.0, 0.0, 4.0],
    [-2.0, -3.0, -1.0],
])

# Validate the matrix shape.
assert X.shape == (4, 3)

# Compute vector lengths safely.
norms = np.linalg.norm(X, axis=1, keepdims=True)
assert np.all(norms > 0)

# Normalize each vector direction.
X_unit = X / norms

# Compute cosine similarity matrix.
cosine_matrix = X_unit @ X_unit.T
cosine_matrix = np.round(cosine_matrix, 3)

# Compare magnitude-sensitive dot products.
linear_matrix = X @ X.T
linear_matrix = np.round(linear_matrix, 1)

# Print compact numerical comparisons.
print("Cosine short-long:", cosine_matrix[0, 1])
print("Cosine short-sports:", cosine_matrix[0, 2])
print("Cosine short-opposite:", cosine_matrix[0, 3])
print("Linear short-long:", linear_matrix[0, 1])
print("Linear short-sports:", linear_matrix[0, 2])

# Plot one cosine similarity heatmap.
fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(cosine_matrix, vmin=-1, vmax=1, cmap="coolwarm")

# Add readable axis labels.
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=35, ha="right")
ax.set_yticklabels(labels)

# Annotate each heatmap cell.
for row in range(len(labels)):
    for col in range(len(labels)):
        ax.text(col, row, cosine_matrix[row, col], ha="center", va="center")

# Finish the single visualization.
ax.set_title("Cosine similarity focuses on direction")
fig.colorbar(im, ax=ax, label="cosine similarity")
plt.tight_layout()
plt.show()



### **2.2. Linear Polynomial Kernel**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_A/image_02_02.jpg?v=1783956262" width="250">



>* Measures alignment in original feature space
>* Best for straight-line similarity patterns

>* Captures feature interactions and curved patterns
>* Higher degrees add detail but risk overfitting

>* Linear keeps geometry; polynomial reveals curved patterns
>* Compare visually and numerically after scaling



In [ ]:
#@title Python Code - Linear Polynomial Kernel

# Compare linear and polynomial similarity kernels.
# Use tiny points for clear visualization.
# Keep outputs short for beginners.

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Create four simple two-feature observations.
labels = ["A", "B", "C", "D"]
X = np.array([[1.0, 1.0], [2.0, 1.0], [1.0, 2.0], [-1.0, 1.0]])

# Validate the tiny matrix shape.
assert X.shape == (4, 2)

# Compute the linear kernel matrix.
linear_kernel = X @ X.T

# Compute polynomial kernels from linear scores.
degree_two = (linear_kernel + 1.0) ** 2
degree_three = (linear_kernel + 1.0) ** 3

# Summarize selected pairwise similarities.
pair_ab = labels.index("A"), labels.index("B")
pair_ad = labels.index("A"), labels.index("D")

# Print compact numerical comparisons.
print("Linear A-B:", round(linear_kernel[pair_ab], 2))
print("Linear A-D:", round(linear_kernel[pair_ad], 2))
print("Poly2 A-B:", round(degree_two[pair_ab], 2))
print("Poly2 A-D:", round(degree_two[pair_ad], 2))
print("Poly3 A-B:", round(degree_three[pair_ab], 2))
print("Poly3 A-D:", round(degree_three[pair_ad], 2))

# Plot three kernel matrices side by side.
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
kernels = [linear_kernel, degree_two, degree_three]
titles = ["Linear", "Polynomial degree 2", "Polynomial degree 3"]

# Draw each heatmap with shared labels.
for ax, matrix, title in zip(axes, kernels, titles):
    sns.heatmap(matrix, annot=True, fmt=".0f", cmap="viridis", xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_title(title)

# Improve spacing before displaying the figure.
plt.tight_layout()
plt.show()



### **2.3. RBF Laplacian Kernels**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_A/image_02_03.jpg?v=1783956260" width="250">



>* Kernels turn distances into similarity scores
>* Nearby points influence each other more

>* Similarity fades as distance increases
>* Kernel width controls local versus broad patterns

>* Capture complex local patterns beyond lines
>* Compare fading similarity and clearer clusters



In [ ]:
#@title Python Code - RBF Laplacian Kernels

# Compare two distance based kernels.
# Small points make matrices readable.
# Heatmaps reveal similarity decay patterns.

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Create tiny two dimensional observations.
points = np.array([[0.0, 0.0], [0.8, 0.2], [2.0, 1.8], [2.4, 2.2], [4.0, 0.5]])
labels = ["A", "B", "C", "D", "E"]

# Validate the small teaching dataset.
assert points.shape == (5, 2)
assert len(labels) == points.shape[0]

# Compute pairwise Euclidean distances manually.
differences = points[:, None, :] - points[None, :, :]
squared_distances = np.sum(differences ** 2, axis=2)
distances = np.sqrt(squared_distances)

# Convert distances into kernel similarities.
gamma = 0.7
rbf_kernel = np.exp(-gamma * squared_distances)
laplacian_kernel = np.exp(-gamma * distances)

# Print compact numerical comparisons.
print("Distance A to B:", round(distances[0, 1], 3))
print("Distance A to E:", round(distances[0, 4], 3))
print("RBF similarity A-B:", round(rbf_kernel[0, 1], 3))
print("RBF similarity A-E:", round(rbf_kernel[0, 4], 3))
print("Laplacian similarity A-B:", round(laplacian_kernel[0, 1], 3))
print("Laplacian similarity A-E:", round(laplacian_kernel[0, 4], 3))

# Plot both kernel matrices side by side.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.heatmap(rbf_kernel, annot=True, cmap="viridis", xticklabels=labels, yticklabels=labels, ax=axes[0], vmin=0, vmax=1)
axes[0].set_title("RBF kernel")

# Show Laplacian similarities using matching scale.
sns.heatmap(laplacian_kernel, annot=True, cmap="viridis", xticklabels=labels, yticklabels=labels, ax=axes[1], vmin=0, vmax=1)
axes[1].set_title("Laplacian kernel")
plt.tight_layout()
plt.show()



## **3. Special Kernels**

### **3.1. Chi Squared Kernel**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_A/image_03_01.jpg?v=1783956271" width="250">



>* Compares nonnegative counts, frequencies, and histograms
>* Highlights proportional category pattern differences

>* Train using pairwise chi squared similarities.
>* Predict by comparing new samples to training.

>* Use only nonnegative, well-scaled histogram features
>* Best for manageable, clearly histogram-based datasets



In [ ]:
#@title Python Code - Chi Squared Kernel

# Demonstrate chi squared histogram similarities.
# Use tiny nonnegative category count data.
# Show precomputed matrices for prediction.

import numpy as np
import matplotlib.pyplot as plt

# Create small histogram-like training examples.
X_train = np.array([
    [8.0, 1.0, 1.0],
    [7.0, 2.0, 1.0],

    [1.0, 8.0, 1.0],
    [1.0, 7.0, 2.0],
])

# Store simple class labels for examples.
y_train = np.array(["red", "red", "green", "green"])

# Create two new histogram-like examples.
X_new = np.array([
    [6.0, 3.0, 1.0],
    [1.0, 6.0, 3.0],
])

# Validate nonnegative histogram inputs.
assert X_train.shape == (4, 3)
assert X_new.shape == (2, 3)
assert np.all(X_train >= 0.0)
assert np.all(X_new >= 0.0)

# Define the additive chi squared distance.
def chi2_distance_matrix(A, B, eps=1e-12):
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)

    numerator = (A[:, None, :] - B[None, :, :]) ** 2
    denominator = A[:, None, :] + B[None, :, :] + eps
    return 0.5 * np.sum(numerator / denominator, axis=2)

# Convert distances into chi squared kernel similarities.
def chi2_kernel_matrix(A, B, gamma=1.0):
    distances = chi2_distance_matrix(A, B)
    return np.exp(-gamma * distances)

# Compute training and prediction kernel matrices.
K_train = chi2_kernel_matrix(X_train, X_train)
K_new = chi2_kernel_matrix(X_new, X_train)

# Validate estimator-compatible matrix shapes.
assert K_train.shape == (len(X_train), len(X_train))
assert K_new.shape == (len(X_new), len(X_train))

# Predict using nearest training similarity.
best_matches = np.argmax(K_new, axis=1)
predicted_labels = y_train[best_matches]

# Print compact numerical teaching output.
print("Training kernel shape:", K_train.shape)
print("Prediction kernel shape:", K_new.shape)
print("New sample predictions:", predicted_labels.tolist())
print("Best training matches:", best_matches.tolist())

# Plot the precomputed training kernel matrix.
fig, ax = plt.subplots(figsize=(5, 4))
image = ax.imshow(K_train, cmap="viridis", vmin=0, vmax=1)

# Label rows and columns clearly.
ax.set_title("Chi Squared Kernel Matrix")
ax.set_xlabel("Training example index")
ax.set_ylabel("Training example index")

# Add a compact colorbar for similarity.
fig.colorbar(image, ax=ax, label="similarity")
plt.tight_layout()
plt.show()



### **3.2. Precomputed Kernel Matrices**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_A/image_03_02.jpg?v=1783956275" width="250">



>* Directly provide pairwise similarities to estimators
>* Connect domain-specific similarity with kernel methods

>* Match matrix format to estimator expectations
>* Compare new samples against training samples

>* Large matrices need valid, scalable similarities
>* Prevent leakage and match estimator requirements



### **3.3. Kernel Limitations**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_A/image_03_03.jpg?v=1783956273" width="250">



>* Kernel matrices need valid similarity structure
>* Invalid kernels can produce misleading model results

>* Kernel matrices grow quickly with dataset size
>* Rich kernels may be too slow for deployment

>* Match matrices to compatible estimators
>* Design kernels with domain understanding



# <font color="#418FDE" size="6.5" uppercase>**Distances Kernels**</font>


In this lecture, you learned to:
- Compute pairwise distances and similarities for small datasets. 
- Compare common distance metrics and kernel functions visually and numerically. 
- Use precomputed distance or kernel matrices with compatible estimators. 

In the next Lecture (Lecture B), we will go over 'Kernel Approximation'